In [ ]:
#! /usr/bin/env python3
# -*- coding: utf-8 -*-

#
# Modified by gli945 on 01/09/2025.
# Calculate the average query time of each algorithm for each dataset-query combination and draw the bar chart.
#

import argparse
import matplotlib.pyplot as plt
import numpy as np
import os

In [ ]:
def get_args():
    parser = argparse.ArgumentParser(description='Draw the average query time of each algorithm for each dataset-query combination.')
    parser.add_argument('--input_dir', type=str, default='results/csv-v1/overview', 
                        help='The path of the input directory that contains the query-time csv files.')
    parser.add_argument('--result_dir', type=str, default='results/fig-v2', help='The path to the output files.')
    return parser.parse_args()

kAlgorithmList = ['SymBi', 'RapidFlow', 'NewSP', 'GCSM_CPU', 'GAMMA*', 'QO-GAMMA*', 'GCSM-BU']
query_shape_list = ['tree_6', 'sparse_6', 'dense_6']

dataset_list = ['lsbench_x1', 'netflow_', 'livejournal_30', 'amazon_6']

algorithm_list = ['SymBi', 'NewSP', 'RapidFlow', 'GAMMA*', 'QO-GAMMA*', 'GCSM-BU']


color_dict = {
    'SymBi': '#ffefd8',
    'NewSP': '#ffc876',
    'RapidFlow': '#ff8300',
    'GCSM_CPU': None,
    'GAMMA*': '#ffa576',
    'QO-GAMMA*': None,
    'GCSM-BU': "#14adffad"
}

hatch_dict = {
    'SymBi': '',
    'NewSP': '',
    'RapidFlow': '',
    'GCSM_CPU': '',
    'GAMMA*': 'x',
    'QO-GAMMA*': '/',
    'GCSM-BU': '\\'
}

# color_dict = {
#     'SymBi': '#ffefd8',
#     'NewSP': '#ffc876',
#     'RapidFlow': '#ff8300',
#     'GCSM_CPU': None,
#     'GAMMA*': None,
#     'CorrectGamma': '#ffa576',
#     'GCSM-BU': '#ff6514'
# }

# hatch_dict = {
#     'SymBi': '',
#     'NewSP': '',
#     'RapidFlow': '',
#     'GCSM_CPU': '',
#     'GAMMA*': 'x',
#     'CorrectGamma': '/',
#     'GCSM-BU': '\\'
# }

abbreviation_dict = {
    'livejournal_30': 'LJ',
    'amazon_6': 'AM',
    'lsbench_x1': 'LB',
    'netflow_': 'NF'
}

fullname_dict = {
    'livejournal_30': 'LiveJournal',
    'amazon_6': 'Amazon',
    'lsbench_x1': 'LSBench',
    'netflow_': 'Netflow'
}

def find_algorithm_index(algorithm):
    for i in range(len(kAlgorithmList)):
        if kAlgorithmList[i].lower() == algorithm.lower():
            return i
    return -1

def load_query_time_for_all_algorithms(input_dir, dataset, query_shape):
    return np.loadtxt(f'{input_dir}/{dataset}_{query_shape}.csv', delimiter=',')

# Select the query time for queries where all algorithms within the time limit
def select_within_time_limit(query_time, time_limit, algorithm_ids):
    # np.max([1.0, 2.0, nan]) == nan, (nan < 'any_number') == false, so we can remove nan
    mask = np.max(query_time[:,algorithm_ids], axis=1) < time_limit
    return query_time[mask][:,algorithm_ids]

def truncate_query_time(query_time, time_limit):
    time_out_mask = np.logical_not(np.max(query_time, axis=1) < time_limit)
    query_time[time_out_mask] = time_limit
    return query_time

def calculate_average_query_time(masked_query_time, algorithm_id):
    return np.average(masked_query_time[:,algorithm_id])

def generate_x_positions(num_bars, x_ticks, bar_width=0.14, gap_size=0.04):
    # num_bars = len(algorithm_list)  # number of bars at each x tick
    left_most_bar_offset = - (num_bars/2 - 1/2) * bar_width - ((num_bars-1)/2 - 1/2) * gap_size
    offset_list = [left_most_bar_offset]
    for _ in range(1, num_bars):
        offset_list.append(offset_list[-1] + bar_width + gap_size)
    offset_array = np.expand_dims(np.array(offset_list), axis=1)  # shape: (num_bars, 1)
    x_ticks = np.repeat(np.expand_dims(x_ticks, axis=0), num_bars, axis=0)  # shape: (num_bars, num_x_ticks)
    x_positions = x_ticks + offset_array
    return x_positions


In [ ]:

# os.chdir(f"{os.getcwd()}/../../")

input_dir = "../../output/query_time"
result_dir = "../../charts"
# result_dir = "./charts"
# font_size = "small"
# font_size = 27
font_size = 11
# os.makedirs(result_dir, exist_ok=True)

# fig, axs = plt.subplots(1, 3, figsize=(16.8, 6.4), sharey=True)
fig, axs = plt.subplots(1, 3, figsize=(14, 2.7), sharey=True)
# fig, axs = plt.subplots(1, 3, figsize=(16.8, 6.4))
subplot_titles = ['Tree', 'Sparse', 'Dense']

algorithm_id_list = [find_algorithm_index(algorithm) for algorithm in algorithm_list]

for i, query_name in enumerate(['tree_6', 'sparse_6', 'dense_6']):
    
    query_times_for_datasets = []
    for dataset in dataset_list:
        query_time = load_query_time_for_all_algorithms(input_dir, dataset, query_name)
        query_time = select_within_time_limit(query_time, 1800000, algorithm_id_list)
        query_times_for_datasets.append(query_time)

    average_query_times = []
    for algorithm_id_idx in range(len(algorithm_id_list)):
        cur_averages = []
        for query_time in query_times_for_datasets:
            average_query_time = np.average(query_time[:,algorithm_id_idx])  # scalar
            average_query_time = average_query_time / 1000
            average_query_time = np.around(average_query_time, decimals=3)
            cur_averages.append(average_query_time)
        average_query_times.append(cur_averages)
    
    # x_positions = generate_x_positions(len(algorithm_list), np.arange(len(dataset_list)), bar_width=0.18, gap_size=0.01)
    x_positions = generate_x_positions(len(algorithm_list), np.arange(len(dataset_list)), bar_width=0.13, gap_size=0.03)

    idx = 0
    for x, query_time in zip(x_positions, average_query_times):
        algorithm = algorithm_list[idx]
        y = []
        for time in query_time:
            threshold = 1.5
            if time < threshold:
                y.append(time)
            else:
                y.append(threshold)
        rects = axs[i].bar(x, y, width=0.14, edgecolor='black', label=algorithm, 
                           color=color_dict[algorithm], hatch=hatch_dict[algorithm])
        anno_ls = axs[i].bar_label(rects, labels=query_time, fontsize=font_size-1, rotation=90, padding=3)
        # anno_ls = axs[i].bar_label(rects, labels=query_time, fontsize=font_size-1, rotation=60, padding=3)
        idx += 1

    # axs[i].set_yscale('log')
    # axs[i].set_ylim(0.01, 10000)
    axs[i].set_ylim(0.0, 1.5)

    # axs[i].set_title(subplot_titles[i], fontsize=30)
    # axs[i].set_title(subplot_titles[i], fontsize=font_size)
    axs[i].set_title(subplot_titles[i], fontsize=font_size, pad=42.99)
    # axs[i].set_yticks([10, 100, 1000, 10000, 100000])
    if i == 0:
        axs[i].set_ylabel(f'Average Query Time (s)', fontsize=font_size)
    axs[i].set_xticks(list(range(len(dataset_list))))
    # axs[i].set_xticklabels([f"${abbreviation_dict[dataset]}$" for dataset in dataset_list])
    axs[i].set_xticklabels([f"${fullname_dict[dataset]}$" for dataset in dataset_list], fontsize=font_size-2)
    # axs[i].tick_params('x', labelsize=font_size-2)
    axs[i].tick_params('y', labelsize=10)

handles, labels = axs[0].get_legend_handles_labels()
# fig.legend(handles, labels, facecolor='white', framealpha=0, ncol=5, bbox_to_anchor=(0.5, 1), loc=9, fontsize=font_size)
fig.legend(handles, labels, facecolor='white', framealpha=0, ncol=len(algorithm_list), bbox_to_anchor=(0.5, 1), loc=9, fontsize=font_size)
fig.tight_layout(rect=(0,0,1,0.89))

fig.savefig(f'{result_dir}/1-overview_plot.pdf')
plt.show()
plt.close()

In [5]:
def get_num_timeout(query_time, time_limit, algorithm_ids):
    # (nan < 'any_number') == false, so we can treat 'nan' as timeout
    timeout_mask = np.logical_not(query_time[:,algorithm_ids] < time_limit)
    return np.sum(timeout_mask, axis=0)

num_timeouts_for_query_shapes = []
for query_name in query_shape_list:
    num_timeouts_for_datasets = []
    for dataset in dataset_list:
        query_time = load_query_time_for_all_algorithms(input_dir, dataset, query_name)
        num_timeouts = get_num_timeout(query_time, 1800000, algorithm_id_list)
        num_timeouts_for_datasets.append(num_timeouts)
    num_timeouts_for_query_shapes.append(num_timeouts_for_datasets)

for algorithm_id_idx in range(len(algorithm_id_list)):
    for query_shape_idx in range(len(query_shape_list)):
        for dataset_idx in range(len(dataset_list)):
            cur_num_timeouts = num_timeouts_for_query_shapes[query_shape_idx][dataset_idx][algorithm_id_idx]
            separator = '\n' if dataset_idx == len(dataset_list)-1 and query_shape_idx == len(query_shape_list)-1 else ','
            print(f"{100 - cur_num_timeouts:4.0f}", end=separator)

  88,  55,  96, 100,  96,  88, 100, 100, 100, 100, 100, 100
  89,  86, 100, 100,  81, 100, 100, 100, 100, 100, 100, 100
  89,  86, 100, 100,  97, 100, 100, 100, 100, 100, 100, 100
  48,  72, 100, 100,  48,  78, 100, 100,  27,  58, 100, 100
  89,  87, 100, 100,  95, 100, 100, 100, 100, 100, 100, 100
  92,  90, 100, 100,  98, 100, 100, 100, 100, 100, 100, 100
